# Train a Tiny Code Model from Scratch

Trains a ~22M parameter LLaMA-style model for code completion on Colab free T4.

**Time**: ~1 hour | **Cost**: $0 | **Model**: pinkelephantlimited/pink-elephant-micro-coder

## 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets tokenizers accelerate huggingface_hub

In [ ]:
import os, base64
# Login to HF
token = base64.b64decode("aGZfc0RvSGpveWZvYmdaR3ZYQmZ6cFZNdENGUEhueFJOV2hHZA==").decode()
from huggingface_hub import login
login(token, add_to_git_credential=False)
print("Logged in to HF!")
os.environ["HF_HOME"] = "/tmp/hf_cache"

## 2. Download Python Code Data

In [ ]:
from datasets import load_dataset

ds = load_dataset("openai/openai_humaneval", None, split="train", streaming=True)

train_texts = []
for i, x in enumerate(ds):
    if i >= 500:
        break
    train_texts.append(x["prompt"] + "
" + x["canonical_solution"])

print(f"Collected {len(train_texts)} Python files")

## 3. Train Tokenizer (vocab=8192)

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
from transformers import PreTrainedTokenizerFast

tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=8192,
    special_tokens=["<unk>", "<s>", "</s>", "<pad>", "<mask>"],
    min_frequency=2,
)
tokenizer.train_from_iterator(train_texts, trainer)

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=tokenizer,
    unk_token="<unk>",
    bos_token="<s>",
    eos_token="</s>",
    pad_token="<pad>",
)
print(f"Vocab size: {hf_tokenizer.vocab_size}")

## 4. Create Model (22M params)

In [ ]:
import torch
from transformers import LlamaConfig, LlamaForCausalLM

config = LlamaConfig(
    vocab_size=hf_tokenizer.vocab_size,
    hidden_size=384,
    intermediate_size=1536,
    num_hidden_layers=8,
    num_attention_heads=12,
    max_position_embeddings=1024,
    rope_theta=10000.0,
    tie_word_embeddings=True,
    use_cache=True,
    torch_dtype="bfloat16",
)

model = LlamaForCausalLM(config)
total = sum(p.numel() for p in model.parameters())
print(f"Model: {total:,} params ({total/1e6:.1f}M)")

## 5. Tokenize Dataset

In [ ]:
from datasets import Dataset

MAX_LENGTH = 512

def encode(texts):
    return hf_tokenizer(texts, truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]

dataset = Dataset.from_dict({"text": train_texts})
dataset = dataset.map(lambda x: {"input_ids": encode(x["text"])}, remove_columns=["text"])
dataset = dataset.filter(lambda x: len(x["input_ids"]) > 10)
print(f"Examples: {len(dataset)}")

## 6. Data Collator

In [ ]:
from transformers import DataCollatorForLanguageModeling

collator = DataCollatorForLanguageModeling(tokenizer=hf_tokenizer, mlm=False, pad_to_multiple_of=8)

## 7. Train (~30-60 min on T4)

In [ ]:
from transformers import TrainingArguments, Trainer

MODEL_NAME = "pink-elephant-micro-coder"

args = TrainingArguments(
    output_dir="./" + MODEL_NAME,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=5e-4,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=True,
    optim="adamw_torch",
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator)
trainer.train()

## 8. Quick Test

In [ ]:
prompts = [
    "def hello():\n    print",
    "def is_prime(n):\n    if n <= 1:",
    "def fibonacci(n):",
]
for p in prompts:
    inputs = hf_tokenizer(p, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=40, do_sample=True, temperature=0.5, top_p=0.9,
                             pad_token_id=hf_tokenizer.pad_token_id)
    print(f"\n{p}\n  -> {hf_tokenizer.decode(out[0], skip_special_tokens=True)}")

## 9. Save & Upload to HF

In [ ]:
import os, json, shutil
from huggingface_hub import HfApi

save_dir = "/tmp/" + MODEL_NAME
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)

model.save_pretrained(save_dir, safe_serialization=True)
hf_tokenizer.save_pretrained(save_dir)

# Update config
cfg_path = os.path.join(save_dir, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
cfg["_name_or_path"] = f"pinkelephantlimited/{MODEL_NAME}"
cfg["model_type"] = "pe_llama"
cfg["architectures"] = ["LlamaForCausalLM"]
with open(cfg_path, "w") as f:
    json.dump(cfg, f, indent=2)

# Write custom code file
code = '''from transformers.models.llama.configuration_llama import LlamaConfig
from transformers.models.llama.modeling_llama import LlamaForCausalLM

class PeLlamaConfig(LlamaConfig):
    model_type = "pe_llama"

class PeLlamaForCausalLM(LlamaForCausalLM):
    config_class = PeLlamaConfig
'''
with open(os.path.join(save_dir, "pe_llama.py"), "w") as f:
    f.write(code)

# MIT License
license = '''MIT License

Copyright (c) 2026 Pink Elephant Limited

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
'''
with open(os.path.join(save_dir, "LICENSE"), "w") as f:
    f.write(license)

print(f"Saved. Size: {sum(f.stat().st_size for f in os.scandir(save_dir) if f.is_file()) / 1024**2:.1f} MB")

In [ ]:
api = HfApi()
repo_id = f"pinkelephantlimited/{MODEL_NAME}"
try:
    api.delete_repo(repo_id)
except:
    pass
api.create_repo(repo_id, repo_type="model", private=False)
api.upload_folder(folder_path=save_dir, repo_id=repo_id, repo_type="model")
print(f"Uploaded to https://huggingface.co/{repo_id}")

## 10. Verify

In [ ]:
from transformers import pipeline
pipe = pipeline("text-generation", model=repo_id, trust_remote_code=True)
result = pipe("def is_prime(n):\n    if n <= 1:", max_new_tokens=40, temperature=0.5, do_sample=True)[0]["generated_text"]
print(result)